In [ ]:
import os
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import pydicom
import matplotlib.pyplot as plt
import pydicom_seg as dcmseg
from radiomics import featureextractor
import pandas as pd

In [ ]:
input_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/0.000000-NA-82046'))
seg_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/300.000000-Segmentation-9.554'))
output_dicom_dir = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/nii_output'))

In [ ]:
# testing stuff with paths

general_dir = Path(os.path.expanduser('~/Documents/Test-NSCLC/'))
path_records = []

for patient_dir in general_dir.iterdir():
    #if not patient_dir.is_dir():
    #    continue

    scan_id = patient_dir.name

    for study_dir in patient_dir.iterdir():
        #if not study_dir.is_dir():
        #    continue

        ct_series = None
        seg_series = None

        for series_dir in study_dir.iterdir():
            if not series_dir.is_dir():
                continue

            if 'Segmentation' in series_dir.name and any(series_dir.glob('*.dcm')):
                seg_series = series_dir
                continue

            if any(series_dir.glob('*.dcm')) and len(list(series_dir.glob('*.dcm'))) >= 10:
                ct_series = series_dir

        if ct_series is not None and seg_series is not None:
            path_records.append({
                'scan_id': scan_id,
                'path_ct': ct_series,
                'path_mask':seg_series
            })
        else:
            print(f"Skipping {patient_dir.name}/{study_dir.name}: ct_series={ct_series is not None}, seg_series={seg_series is not None}")

path_df = pd.DataFrame(path_records, columns=['scan_id', 'path_ct', 'path_mask'])

In [ ]:
#input_dcm_new = sorted(input_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts
#seg_dcm_new = sorted(seg_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts

ser_reader = sitk.ImageSeriesReader()
i = 0
for ct_path, mask_path in zip(path_df['path_ct'], path_df['path_mask']):
    #seg_file = sorted(mask_path.glob('*.dcm'))

    #find correct file and read it as a dicom segmentation object
    seg = pydicom.dcmread(list(mask_path.glob('*.dcm'))[0])
    seg_reader = dcmseg.SegmentReader()
    result = seg_reader.read(seg)

    # find segmentation from neoplasm label
    seg_infos = result.segment_infos
    for seg_num, info in seg_infos.items():
        
        if 'Neoplasm' not in info.get('SegmentLabel', ''):
            neo_seg_num = seg_num
        else:
            print(f"Found neoplasm segment: {seg_num} with label {info['SegmentLabel']} in mask: {mask_path}")
            continue

                
    #segment_sequence = result.available_segments
    #print(segment_sequence)

    #for segment in segment_sequence:
    #    print(seg.SegmentSequence[segment - 1].SegmentLabel)


In [ ]:
dcm = pydicom.dcmread(input_dcm_new[70]) # takes first image
sitk_img = sitk.ReadImage(input_dcm_new[70]) # reads the dicom series as a sitk image


#You should preferably use the series reader!!!
series_reader = sitk.ImageSeriesReader()
files = series_reader.GetGDCMSeriesFileNames(input_dcm)
series_reader.SetFileNames(files)
sitk_imgs = series_reader.Execute() # reads the dicom series as a sitk image

In [ ]:
ct_arr = sitk.GetArrayFromImage(sitk_imgs)
print(ct_arr.shape)

In [ ]:
z = 80 # this is the slice number

arr = sitk.GetArrayFromImage(sitk_imgs)
slice = arr[z, :, :]
plt.imshow(slice, cmap='gray')
plt.show()

In [ ]:
#now do the same with the segmentation dicom...

dcm_seg = pydicom.dcmread(seg_dcm_new[0])

In [ ]:
reader = dcmseg.SegmentReader()
result = reader.read(dcm_seg)
print(result.available_segments)
print(dcm_seg.SegmentSequence[0].SegmentLabel) # you need to make sure that the neoplasm label is always in the same spot!
# in this case label 1 = neoplasm, but is this always tje case?
#Are there different labels than neoplasm that correspond to the subtype of the tumour?

In [ ]:
neoplasm_segment = result.segment_data(1) # this is the neoplasm segment, but is this always the case?
print(neoplasm_segment.shape)
neoplasm_segment_img = result.segment_image(1) # or should I do this..?
img_array_neoplasm = sitk.GetArrayFromImage(neoplasm_segment_img)
print(img_array_neoplasm.shape)
print(len(input_dcm_new))

plt.imshow(slice, cmap='gray') # overlay the original image with the segmentation mask
plt.imshow(img_array_neoplasm[80], cmap='gray', alpha=0.2) # overlay the segmentation mask with the original image
plt.show()
#something is wrong with the segmentation mask, it doesn't fit the original ct

In [ ]:
print(dcm.SeriesInstanceUID)
print(dcm_seg.ReferencedSeriesSequence[0].SeriesInstanceUID)
print(dcm.GantryDetectorTilt)
print(dcm_seg.SharedFunctionalGroupsSequence[0]
      .PixelMeasuresSequence[0].PixelSpacing, dcm.PixelSpacing)
print(dcm_seg.SharedFunctionalGroupsSequence[0]
      .PixelMeasuresSequence[0].SliceThickness, dcm.SliceThickness)


In [ ]:
def initialize_feature_extractor():
    paramsFile = "CEM_extraction.yaml"
    extractor = featureextractor.RadiomicsFeatureExtractor(paramsFile, shape2D=True, force2D=True,
                                                               force2Ddimension=True, resampledPixelSpacing=None)
    extractor.addProvenance(False)
    extractor.disableAllFeatures()
    extractor.enableImageTypes(Original={})

    extractor.enableFeatureClassByName('firstorder', enabled=True)
    extractor.enableFeatureClassByName('shape2D', enabled=True)
    extractor.enableFeatureClassByName('glcm', enabled=True)
    extractor.enableFeatureClassByName('glrlm', enabled=True)
    extractor.enableFeatureClassByName('glszm', enabled=True)
    extractor.enableFeatureClassByName('gldm', enabled=True)
    extractor.enableFeatureClassByName('ngtdm', enabled=True)
    return extractor

In [ ]:
extractor = initialize_feature_extractor()
fixed_seg = sitk.Cast(neoplasm_segment_img, sitk.sitkUInt8)
fixed_seg.CopyInformation(sitk_imgs)

In [ ]:
def extract_slice(img, slice_no):
    size = list(img.GetSize())
    index = [0, 0, slice_no]

    size[2] = 0  # extract 2D slice
    return sitk.Extract(img, size, index)

In [ ]:
for slice_no in range(80, 85):
    img_slice = extract_slice(sitk_imgs, slice_no)
    seg_slice = extract_slice(fixed_seg, slice_no)

    # Check if segmentation contains label 1
    if 1 not in sitk.GetArrayViewFromImage(seg_slice):
        continue

    features = extractor.execute(img_slice, seg_slice, label=1)
    print(f" features:", features)

In [ ]:
features

In [ ]:

result_extraction = extractor.execute(sitk_imgs, neoplasm_segment_img)

In [ ]:
input_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/0.000000-NA-82046'))
seg_dcm = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/300.000000-Segmentation-9.554'))
output_dicom_dir = Path(os.path.expanduser('~/Documents/Test-NSCLC/LUNG1-001/09-18-2008-StudyID-NA-69331/nii_output'))
series_IDs = sitk.ImageSeriesReader.GetGDCMSeriesIDs(input_dcm)

itk_images = []
for i in range(0,len(series_IDs)):
  series_file_names = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(input_dcm, series_IDs[i])
  series_reader = sitk.ImageSeriesReader()
  series_reader.SetFileNames(series_file_names)
  series_reader.MetaDataDictionaryArrayUpdateOn()
  series_reader.LoadPrivateTagsOn()
  image_dicom = series_reader.Execute()
  
  itk_images.append(image_dicom)
  sitk.WriteImage(image_dicom, os.path.join(output_dicom_dir, series_IDs[i] + ".nii.gz"))

In [ ]:
def generate_features_table(df, extractor,inference_usage=False):
    # warning: this function can take a long time to run
    # extract low energy features
    featureVector_low_energy = extractor.execute(list(df["path_low_energy"])[0], list(df["path_mask"])[0])
    temp_dataset = pd.Series(featureVector_low_energy)
    feature_df_low_energy = pd.DataFrame([temp_dataset], columns=list(featureVector_low_energy.keys()),
                                         index=[list(df["path_mask"])[0]])
    for i, temp_mask in tqdm.tqdm(enumerate(list(df["path_mask"])[1:])):
        featureVector_low_energy = extractor.execute(list(df["path_low_energy"])[i + 1], temp_mask)
        temp_dataset = pd.Series(featureVector_low_energy)
        feature_df_low_energy.loc[temp_mask] = temp_dataset.values
    feature_df_low_energy.columns = feature_df_low_energy.columns + "_low_energy"